# Week 4 Task: Machine Learning Model Development & Evaluation
## Customer Churn Prediction Pipeline & Performance Benchmarking

This notebook demonstrates the end-to-end machine learning pipeline from data loading, preprocessing, model selection, training, evaluation, to visualization generation.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

### 1. Load Dataset
Load the customer dataset generated for the classification task.

In [ ]:
df = pd.read_csv('customer_churn_data.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

### 2. Preprocessing & Pipeline Building
Construct ColumnTransformer for numerical scaling and categorical encoding.

In [ ]:
feature_cols = ['age', 'tenure_months', 'monthly_spend', 'total_purchases', 'support_tickets', 'discount_usage_pct', 'contract_type', 'device_type']
X = df[feature_cols]
y = df['churn']

numeric_features = ['age', 'tenure_months', 'monthly_spend', 'total_purchases', 'support_tickets', 'discount_usage_pct']
categorical_features = ['contract_type', 'device_type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ]
)
print("Preprocessing pipeline successfully defined.")

### 3. Model Training & Benchmarking
Train Logistic Regression, Decision Tree, and Random Forest classifiers.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
}

results = {}
for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    }

results_df = pd.DataFrame(results).T
display(results_df)

### 4. Performance Visualizations
Display model evaluation charts.

In [ ]:
results_df.plot(kind='bar', figsize=(10, 6))
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.ylim(0.5, 1.0)
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()